# 🧬 Persona LoRA — fine-tune a small model to text like Bohdan

Closes the **lexical-voice gap** the RAG prompt can't reach: uk/en/ru code-switch,
opener variety, the `)` smiley tic, real casing, multi-bubble bursts. Free **Colab T4**.

**Base:** Qwen2.5-3B-Instruct (best Cyrillic tokenizer of the small models). QLoRA 4-bit,
via Unsloth. End to end: ~20–40 min on a T4.

### The one architectural rule: train == serve
The adapter is trained on a **thin** prompt — a 1-line persona anchor + the joined
incoming context — with the loss flowing only through *your* reply. So this notebook
**evaluates and serves it under that same thin shape**, never the heavy RAG template.
Feeding a 3B LoRA a 1600-token English instruction wall at serving time drags it back
to its generic assistant register and undoes the fine-tune (the audit's #1 finding).

### Before you start
1. `Runtime → Change runtime type → T4 GPU`.
2. Locally: `uv run python scripts/export_finetune_data.py` → upload
   `data/finetune/train.jsonl` **and** `eval.jsonl` in the Upload cell below.

> **Honest target** (printed by the export): your replies are register-matched across
> train/eval — `latin_script_rate ≈ 0.18`, `paren_smiley_rate ≈ 0.05`,
> `opener_top_share ≈ 0.06`. The old `0.468` latin goal was a temporal-split artifact;
> don't chase it. The real skill is **per-context mirroring** (English back to the
> English contact, Ukrainian to everyone else), which a context→reply LoRA learns for free.

## 1. Install Unsloth (with a hard API check)
Colab ships an **older `transformers` than the latest Unsloth needs** (and an older
`protobuf`), so we upgrade them *together* — installing only Unsloth leaves Colab's
stale `transformers`, which fails with `cannot import name 'PreTrainedConfig'`, and the
stale `protobuf` fails with `cannot import name 'runtime_version'`. After the upgrade
you **must restart the runtime** (next cell). The cell after that **fails loudly** if
the APIs this notebook needs ever move.

In [ ]:
%%capture
# Upgrade Unsloth AND the libs Colab pre-pins too old for it, in one resolve.
!pip install -q --upgrade unsloth unsloth_zoo transformers trl peft protobuf

### ⚠️ Restart the runtime now, then continue below
The upgrade replaced `transformers`/`protobuf`, which Colab had **already imported**.
Do **`Runtime → Restart session`**, then run **from the version-check cell below** —
do *not* re-run the install cell. This is what clears the `runtime_version` /
`PreTrainedConfig` import errors.

In [ ]:
import importlib
for m in ["unsloth", "trl", "transformers", "torch", "peft"]:
    try:
        print(f"{m:13s}", importlib.import_module(m).__version__)
    except Exception as e:
        print(f"{m:13s} MISSING:", e)
# Hard guard: the exact APIs this notebook relies on must exist.
from unsloth.chat_templates import (  # noqa: F401
    standardize_sharegpt,
    get_chat_template,
    train_on_responses_only,
)
print("\nOK — unsloth chat-template APIs present")

## 2. Upload your data
Run, then pick `train.jsonl` **and** `eval.jsonl`.

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose train.jsonl AND eval.jsonl
assert 'train.jsonl' in uploaded, 'upload train.jsonl'
assert 'eval.jsonl' in uploaded, 'upload eval.jsonl (needed for the voice eval)'
print(list(uploaded.keys()))

## 3. Load Qwen2.5-3B-Instruct in 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

# 1536 covers the export's 2000-char context cap + reply without truncating the
# response (train_on_responses_only needs the reply tokens), and is ~25% leaner on
# the T4 than 2048.
MAX_SEQ_LEN = 1536
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,
    dtype = None,
)

## 4. Attach the LoRA adapter
`r=32`, **`alpha=64` (2·r)** — a stronger update because the whole job is *overriding*
the base model's polite/monolingual/Title-Case defaults. `dropout=0` keeps Unsloth's
fused fast path (no benefit from dropout on 20k clean pairs).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    lora_alpha = 64,          # 2*r: push hard enough to override the base register
    lora_dropout = 0,         # 0 keeps Unsloth's fused kernel; not needed here
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 5. Build the dataset
ShareGPT → Qwen chat template. The records already carry the thin
`[system: persona anchor][human: joined context][gpt: your reply]` shape.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt, get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

ds = load_dataset("json", data_files="train.jsonl", split="train")
ds = standardize_sharegpt(ds)

def _fmt(batch):
    texts = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in batch["conversations"]
    ]
    return {"text": texts}

ds = ds.map(_fmt, batched=True)
print("examples:", len(ds))
print(ds[0]["text"][:500])

## 6. Trainer — loss on responses only, T4-safe batch
`train_on_responses_only` makes the loss flow **only through your reply tokens** — the
single most important switch for style mimicry. `bs=4 × ga=4` = effective batch 16 with
~half the worst-case activation memory of `bs=8` (no late-epoch OOM). `save_strategy
= epoch` keeps both epoch checkpoints so you can pick the better one by the voice eval.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LEN,
    packing = False,            # required: train_on_responses_only masks per-example
    args = SFTConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,   # effective batch 16
        warmup_ratio = 0.05,
        num_train_epochs = 2,              # 2 is plenty; more parrots. Pick best by eval.
        learning_rate = 2e-4,
        logging_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        save_strategy = "epoch",
        save_total_limit = 2,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

### Sanity-check the response mask
You should see the **context masked out** (`-100` → blank) and only *your reply* kept
as labels. If the whole thing is unmasked, the loss is modelling other people's
messages and the voice won't transfer.

In [ ]:
# Some Unsloth versions add `labels` to the dataset eagerly; others mask in the
# collator at batch time. Handle both so this never errors the run.
ex = trainer.train_dataset[0]
if "labels" in ex:
    kept = [t for t in ex["labels"] if t != -100]
else:
    batch = trainer.data_collator([ex])
    lab = batch["labels"][0].tolist()
    kept = [t for t in lab if t != -100]
print("kept label tokens (should be ~your reply only):")
print(tokenizer.decode(kept))

## 7. Train

In [ ]:
trainer_stats = trainer.train()

In [ ]:
import gc
# free optimizer/grad state so the eval generations below have headroom on the T4
gc.collect()
torch.cuda.empty_cache()

## 8. In-notebook voice eval — *see numbers before you export*
Generates on the held-out `eval.jsonl` contexts **under the thin training-native
prompt** and scores the distribution against your real eval replies. This is the
selection signal — keep the checkpoint whose numbers sit closest to the reference,
**not** the lowest train loss.

In [ ]:
# --- self-contained voice metrics (mirror persona_rag/eval/distribution.py) ---
import re, math, bisect, json as _json
from collections import Counter
from statistics import median

_WORD = re.compile(r'[^\W\d_]+', re.UNICODE)
_LATIN = re.compile(r'[a-z]')
_CYR = re.compile(r'[Ѐ-ӿ]')
_PAREN_SMILEY = re.compile(r'\)\)+|[^\s(]\)(?!\w)')

def split_bubbles(t):
    if not t: return []
    n = t.replace('\\n','\n').replace('\r\n','\n')
    return [c.strip() for c in n.split('\n') if c.strip()]

def _toks(t): return _WORD.findall(t.lower())

def latin_rate(texts):
    toks = [w for x in texts for w in _toks(x)]
    lat = sum(1 for w in toks if _LATIN.search(w))
    cyr = sum(1 for w in toks if _CYR.search(w))
    return lat/(lat+cyr) if (lat+cyr) else 0.0

def opener_top(texts):
    op = []
    for x in texts:
        b = split_bubbles(x)
        if b and _toks(b[0]): op.append(_toks(b[0])[0])
    return Counter(op).most_common(1)[0][1]/len(op) if op else 0.0

def paren_rate(texts):
    bs = [b for t in texts for b in split_bubbles(t)]
    return sum(1 for b in bs if _PAREN_SMILEY.search(b))/len(bs) if bs else 0.0

# caps_first_rate: share of replies whose FIRST letter is uppercase (phone
# autocapitalize). NOTE: this is NOT the API scorecard's per-char `caps_ratio`.
def caps_first_rate(texts):
    n=c=0
    for t in texts:
        b = split_bubbles(t)
        if not b: continue
        m = re.search(r'[^\W\d_]', b[0])
        if not m: continue
        n += 1; c += 1 if m.group().isupper() else 0
    return c/n if n else 0.0

def shape_hist(texts, mx=6):
    cnt={}; n=0
    for t in texts:
        c=len(split_bubbles(t))
        if c==0: continue
        b=min(c,mx); cnt[b]=cnt.get(b,0)+1; n+=1
    return {b: cnt.get(b,0)/n for b in range(1,mx+1)} if n else {}

def js_div(p,q):
    keys=set(p)|set(q); m={k:(p.get(k,0)+q.get(k,0))/2 for k in keys}
    def kl(x): return sum(x[k]*math.log2(x[k]/m[k]) for k in keys if x.get(k,0)>0 and m.get(k,0)>0)
    return 0.5*kl(p)+0.5*kl(q)

def bubble_lens(texts): return [len(b) for t in texts for b in split_bubbles(t)]

def wass(a,b):
    if not a or not b: return 0.0
    sa,sb=sorted(a),sorted(b); pts=sorted(set(sa)|set(sb)); tot=0.0
    for i in range(len(pts)-1):
        ca=bisect.bisect_right(sa,pts[i])/len(sa); cb=bisect.bisect_right(sb,pts[i])/len(sb)
        tot+=abs(ca-cb)*(pts[i+1]-pts[i])
    return tot

def summarize(texts):
    return dict(latin=latin_rate(texts), paren=paren_rate(texts),
                opener_top=opener_top(texts), caps_first_rate=caps_first_rate(texts),
                hist=shape_hist(texts), lens=bubble_lens(texts))

# reference = your real held-out replies
eval_records = [_json.loads(l) for l in open('eval.jsonl', encoding='utf-8')]
def _human(rec): return next(t['value'] for t in rec['conversations'] if t['from']=='human')
def _gpt(rec):   return next(t['value'] for t in rec['conversations'] if t['from']=='gpt')
REF = summarize([_gpt(r) for r in eval_records])
print('reference (your real eval replies):',
      {k: round(REF[k],3) for k in ('latin','paren','opener_top','caps_first_rate')})

In [ ]:
# --- generate under the THIN training-native prompt, then score ---
import random
FastLanguageModel.for_inference(model)
SYSTEM = "Ти Богдан. Пиши так, як ти зазвичай пишеш у телеграмі."

def gen_thin(ctx, max_new_tokens=128):
    msgs = [{"role":"system","content":SYSTEM}, {"role":"user","content":ctx}]
    ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                        return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.8, top_p=0.95, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

def voice_eval(n=64, seed=0):
    rng = random.Random(seed)
    sample = rng.sample(eval_records, min(n, len(eval_records)))
    gen = [gen_thin(_human(r)) for r in sample]
    real = [_gpt(r) for r in sample]
    g = summarize(gen); rf = summarize(real)
    shape_js = js_div(rf['hist'], g['hist'])
    len_w = wass(rf['lens'], g['lens'])
    dist = (shape_js + abs(g['latin']-rf['latin']) + abs(g['paren']-rf['paren'])
            + abs(g['opener_top']-rf['opener_top']) + len_w/50.0)
    print(f'  n={len(sample)}  VOICE_DISTANCE={dist:.3f}  (lower = more like Bohdan)')
    print(f'  shape_js {shape_js:.3f}   len_wass {len_w:.1f}')
    for k in ('latin','paren','opener_top','caps_first_rate'):
        print(f'  {k:15s} gen={g[k]:.3f}  real={rf[k]:.3f}')
    return dist, g, gen, real

# n=64 is a stable read of these rates; takes a few minutes (sequential decoding).
dist, g, gen, real = voice_eval(n=64)
print('\nsamples:')
for ctx_r, gg in list(zip(real, gen))[:8]:
    print(' real:', repr(ctx_r[:60]), '| gen:', repr(gg[:60]))

### (optional) Compare the two epoch checkpoints
Selection by *distribution*, not loss. Reloads each saved epoch and re-scores; keep
the lower `VOICE_DISTANCE`. Skip if the final-model numbers above already look right.

In [ ]:
import glob, os
from peft import PeftModel
ckpts = sorted(glob.glob('outputs/checkpoint-*'))
print('checkpoints:', ckpts)
# To score one: reload base, attach the checkpoint adapter, run voice_eval, then
# re-attach your chosen one before export. Example (uncomment to use):
# base, tok = FastLanguageModel.from_pretrained('unsloth/Qwen2.5-3B-Instruct',
#     max_seq_length=MAX_SEQ_LEN, load_in_4bit=True, dtype=None)
# base = PeftModel.from_pretrained(base, ckpts[0]); FastLanguageModel.for_inference(base)
# (then temporarily point gen_thin at `base`)

## 9. Qualitative probes (thin prompt)
Watch for: real casing, code-switch, the `)` tic, short bursts, and — on the vulnerable
probe — that it **engages** instead of brushing off. No exclamation marks.

In [ ]:
PROBES = [
    "шо там по плану на вечір",
    "сам ти даун шо ти несеш",
    "***REMOVED***, не знаю що робити чесно",
    "yo you coming tonight or nah",
]
for p in PROBES:
    print("IN :", p)
    print("OUT:", gen_thin(p))
    print("-" * 60)

## 10. Export — adapter FIRST (insurance), then GGUF
The LoRA adapter is tiny and always saves; the GGUF convert is the fragile step. Save
the adapter and download it **before** attempting GGUF, so a convert failure never
costs you the 20–40 min run.

In [ ]:
# (a) Insurance: save + zip the adapter. Re-mergeable any time, costs seconds.
model.save_pretrained("lora-adapter")
tokenizer.save_pretrained("lora-adapter")
!cd lora-adapter && zip -r ../bohdan-lora-adapter.zip . >/dev/null
from google.colab import files
files.download("bohdan-lora-adapter.zip")
print("adapter saved + downloaded — your run is now safe")

In [ ]:
# (b) GGUF for Ollama. q5_k_m preserves the subtle voice signal better than q4 and
#     still runs anywhere locally (~2.4 GB). Use q8_0 for max fidelity, q4_k_m to save space.
QUANT = 'q5_k_m'
gguf = None
try:
    model.save_pretrained_gguf("model-gguf", tokenizer, quantization_method=QUANT)
    import os
    gguf = [f for f in os.listdir('model-gguf') if f.endswith('.gguf')][0]
    print('GGUF:', gguf)
except Exception as e:
    print('GGUF convert failed:', repr(e)[:200])
    print('Fallback → saving merged 16-bit weights; convert to GGUF locally with llama.cpp')
    model.save_pretrained_merged("model-merged-16bit", tokenizer, save_method="merged_16bit")
    !cd model-merged-16bit && zip -r ../bohdan-merged-16bit.zip . >/dev/null
    files.download("bohdan-merged-16bit.zip")

In [ ]:
# (c) Modelfile — MUST match training: the thin Ukrainian SYSTEM and the Qwen template.
#     num_ctx 4096 stops Ollama's 2048 default from truncating; num_predict caps runaway
#     bursts (train p99 reply ~254 chars); repeat_penalty guards loops.
assert gguf, 'no GGUF produced — use the merged-16bit fallback locally instead'
modelfile = f'''FROM ./{gguf}
TEMPLATE """{{{{ if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}{{{{ if .Prompt }}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
{{{{ .Response }}}}<|im_end|>
"""
PARAMETER num_ctx 4096
PARAMETER num_predict 256
PARAMETER temperature 0.8
PARAMETER top_p 0.95
PARAMETER repeat_penalty 1.1
PARAMETER stop "<|im_end|>"
SYSTEM """Ти Богдан. Пиши так, як ти зазвичай пишеш у телеграмі."""
'''
open("model-gguf/Modelfile", "w").write(modelfile)
print(modelfile)

In [ ]:
# (d) Zip + download the GGUF bundle for local Ollama (only if the convert
#     succeeded — otherwise use the merged-16bit zip from cell (b)).
if gguf:
    !cd model-gguf && zip -r ../bohdan-lora-gguf.zip . >/dev/null
    files.download("bohdan-lora-gguf.zip")
else:
    print('No GGUF — use bohdan-merged-16bit.zip and convert with llama.cpp locally.')

## 11. Plug into the bot (local machine)
```bash
unzip bohdan-lora-gguf.zip -d bohdan-gguf && cd bohdan-gguf
ollama create bohdan -f Modelfile
ollama run bohdan 'шо там'        # smoke test
```
Then in the repo `.env`:
```
GENERATION_BACKEND=ollama
OLLAMA_MODEL=bohdan
```
`build_messages` now auto-switches to the **thin** prompt for the ollama backend, so
`eval_persona.py` grades the LoRA in its native condition (same shape it trained on):
```bash
uv run python scripts/eval_persona.py --n 200 --name lora-v1
```
**Keep the checkpoint with the lowest `VOICE_DISTANCE` / closest distribution** —
`latin ≈ 0.18`, `paren ≈ 0.05`, `opener_top ≈ 0.06`, low `shape_js` — **not** the
lowest train loss. If it still drifts generic, run the ORPO stage in
`docs/finetune/README.md`.